In [11]:
# ============================================================
# 課題
# アカウント設定ページの自己紹介文を更新する
# ============================================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from getpass import getpass
import time


# ============================================================
# 1. Chromeの設定
# ============================================================

options = webdriver.ChromeOptions()

options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")


# ============================================================
# 2. Chromeを起動
# ============================================================

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 30)

print("Chrome起動成功")


# ============================================================
# 3. TERAKOYAを開く
# ============================================================

driver.get(
    "https://terakoya.sejuku.net/register"
)


# ============================================================
# 4. ログイン画面を開く
# ============================================================

login_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.CSS_SELECTOR,
            "#root > header > div.sc-dmgCwU.gmWXAg"
        )
    )
)

login_button.click()

print("ログイン画面を開きました")


# ============================================================
# 5. メールアドレスとパスワードを入力
# ============================================================

email = input(
    "メールアドレスを入力してください: "
)

password = getpass(
    "パスワードを入力してください: "
)


# ============================================================
# 6. ログインフォームを取得
# ============================================================

parent = wait.until(
    EC.presence_of_element_located(
        (
            By.CSS_SELECTOR,
            ".sc-dFVmKS.ehzDTg"
        )
    )
)


email_input = parent.find_element(
    By.NAME,
    "email"
)

password_input = parent.find_element(
    By.NAME,
    "password"
)


# ============================================================
# 7. ログイン情報を入力
# ============================================================

email_input.clear()
email_input.send_keys(email)

password_input.clear()
password_input.send_keys(password)

time.sleep(2)


# ============================================================
# 8. ログインボタンをクリック
# ============================================================

form_login_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[text()='ログイン']"
        )
    )
)

form_login_button.click()

print("ログインボタンをクリックしました")


# ============================================================
# 9. ★ログイン完了をURLで確認
# ============================================================

wait.until(
    EC.url_contains("/home")
)

print("ログイン成功")
print("現在のURL:", driver.current_url)


# 少し待つ
time.sleep(3)


# ============================================================
# 10. アカウント設定ページへ移動
# ============================================================

driver.get(
    "https://terakoya.sejuku.net/account/profile"
)

print("アカウント設定ページへ移動しました")


# ============================================================
# 11. ページ読み込みを待つ
# ============================================================

time.sleep(5)

print("現在のURL:", driver.current_url)
print("ページタイトル:", driver.title)


# ============================================================
# 12. 403になっていないか確認
# ============================================================

body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text

if "403" in body_text or "Forbidden" in body_text:

    driver.save_screenshot(
        "profile_403.png"
    )

    driver.quit()

    raise Exception(
        "プロフィールページが403 Forbiddenになりました"
    )


print("プロフィールページ表示成功")


# ============================================================
# 13. 自己紹介欄を探す
#
# labelの文字「自己紹介」を基準に探します。
# CSSクラス名のような変更されやすい値に
# なるべく依存しないようにしています。
# ============================================================

profile_input = None


# ------------------------------------------------------------
# 方法1
# textareaを取得
# ------------------------------------------------------------

textareas = driver.find_elements(
    By.TAG_NAME,
    "textarea"
)

if len(textareas) > 0:

    profile_input = textareas[0]

    print(
        f"textareaを発見しました：{len(textareas)}個"
    )


# ------------------------------------------------------------
# 方法2
# textareaで見つからなければ
# 「自己紹介」という文字の近くを探す
# ------------------------------------------------------------

if profile_input is None:

    try:

        profile_input = driver.find_element(
            By.XPATH,
            "//*[contains(text(),'自己紹介')]/following::textarea[1]"
        )

        print(
            "「自己紹介」の近くにtextareaを発見しました"
        )

    except:
        pass


# ------------------------------------------------------------
# 方法3
# contenteditable要素も確認
# ------------------------------------------------------------

if profile_input is None:

    editable_elements = driver.find_elements(
        By.CSS_SELECTOR,
        "[contenteditable='true']"
    )

    if len(editable_elements) > 0:

        profile_input = editable_elements[0]

        print(
            "contenteditable要素を発見しました"
        )


# ============================================================
# 14. 見つからなかった場合
# ============================================================

if profile_input is None:

    print()
    print("自己紹介欄が見つかりませんでした。")

    print()
    print("ページの表示内容:")
    print("-" * 50)

    print(body_text[:5000])

    driver.save_screenshot(
        "profile_not_found.png"
    )

    driver.quit()

    raise Exception(
        "自己紹介欄を特定できませんでした"
    )


# ============================================================
# 15. 指定された自己紹介文
# ============================================================

new_profile = (
    "プログラミング学習中です！"
    "今はスクレイピングに挑戦しています！"
)


# ============================================================
# 16. 現在の自己紹介を削除
# ============================================================

profile_input.clear()


# ============================================================
# 17. 新しい自己紹介文を入力
# ============================================================

profile_input.send_keys(
    new_profile
)

print()
print("自己紹介文を入力しました")

print(
    "入力内容:",
    new_profile
)


# ============================================================
# 18. 更新ボタンを探す
# ============================================================

update_button = None


# 「更新」ボタン
try:

    update_button = driver.find_element(
        By.XPATH,
        "//button[contains(., '更新')]"
    )

except:
    pass


# 「保存」ボタン
if update_button is None:

    try:

        update_button = driver.find_element(
            By.XPATH,
            "//button[contains(., '保存')]"
        )

    except:
        pass


# submitボタン
if update_button is None:

    try:

        update_button = driver.find_element(
            By.CSS_SELECTOR,
            "button[type='submit']"
        )

    except:
        pass


# ============================================================
# 19. 更新ボタンが見つからない場合
# ============================================================

if update_button is None:

    driver.save_screenshot(
        "update_button_not_found.png"
    )

    driver.quit()

    raise Exception(
        "更新ボタンが見つかりませんでした"
    )


# ============================================================
# 20. 更新ボタンをクリック
# ============================================================

driver.execute_script(
    "arguments[0].scrollIntoView({block: 'center'});",
    update_button
)

time.sleep(1)

update_button.click()

print("更新ボタンをクリックしました")


# ============================================================
# 21. 更新処理を待つ
# ============================================================

time.sleep(5)


# ============================================================
# 22. 結果を確認
# ============================================================

print()
print("=" * 60)
print("課題実行結果")
print("=" * 60)

print()

print("設定した自己紹介文:")

print(
    new_profile
)

print()

print(
    "現在のURL:",
    driver.current_url
)


# ============================================================
# 23. 証拠用スクリーンショット
# ============================================================

driver.save_screenshot(
    "profile_updated.png"
)

print()
print(
    "profile_updated.png を保存しました"
)


# ============================================================
# 24. Chrome終了
# ============================================================

driver.quit()

print()
print("処理完了")

Chrome起動成功
ログイン画面を開きました
メールアドレスを入力してください: wisely1015@gmail.com
パスワードを入力してください: ··········
ログインボタンをクリックしました
ログイン成功
現在のURL: https://terakoya.sejuku.net/home
アカウント設定ページへ移動しました
現在のURL: https://terakoya.sejuku.net/account/profile
ページタイトル: プロフィール | プログラミング学習サイト【侍テラコヤ】
プロフィールページ表示成功

自己紹介欄が見つかりませんでした。

ページの表示内容:
--------------------------------------------------
ホーム
タイムライン
教材
課題
Q&A
学習ログ
カリキュラム
専属レッスン
セッション
メッセージ
よくある質問
ご意見・ご要望
侍テラコヤの使い方
侍エンジニアブログ
アカウント
学びたいトピックを検索　例：Laravel、Webサイト制作、学習方法
7
プロフィール
マンツーマン契約
加入プラン
請求情報
購入履歴
支払方法
メール設定
パスワード設定
編集
ユーザー情報
プロフィール画像
受講生ID
15523
メールアドレス
wisely1015@gmail.com
名前
伊藤 淳
ニックネーム
(公開用)
あっし
プログラミング経験
学習予定の言語 /
フレームワーク
自己紹介
三豊市役所から一般社団法人みとよAI社会推進機構（MAiZM）に派遣されています。
学習目標
未登録
© SAMURAI Inc.
利用規約
プライバシーポリシー
運営会社


Exception: 自己紹介欄を特定できませんでした

In [12]:
# ============================================================
# TERAKOYA
# プロフィールの自己紹介文を更新する
# ============================================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from getpass import getpass
import time


# ============================================================
# 1. Chrome設定
# ============================================================

options = webdriver.ChromeOptions()

options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")


# ============================================================
# 2. Chrome起動
# ============================================================

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 30)

print("Chrome起動成功")


# ============================================================
# 3. TERAKOYAを開く
# ============================================================

driver.get(
    "https://terakoya.sejuku.net/register"
)


# ============================================================
# 4. ログイン画面を開く
# ============================================================

login_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.CSS_SELECTOR,
            "#root > header > div.sc-dmgCwU.gmWXAg"
        )
    )
)

login_button.click()

print("ログイン画面を開きました")


# ============================================================
# 5. メールアドレス・パスワード入力
# ============================================================

email = input(
    "メールアドレスを入力してください: "
)

password = getpass(
    "パスワードを入力してください: "
)


# ============================================================
# 6. ログインフォーム取得
# ============================================================

parent = wait.until(
    EC.presence_of_element_located(
        (
            By.CSS_SELECTOR,
            ".sc-dFVmKS.ehzDTg"
        )
    )
)

email_input = parent.find_element(
    By.NAME,
    "email"
)

password_input = parent.find_element(
    By.NAME,
    "password"
)


email_input.clear()
email_input.send_keys(email)

password_input.clear()
password_input.send_keys(password)

time.sleep(2)


# ============================================================
# 7. ログイン
# ============================================================

form_login_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[text()='ログイン']"
        )
    )
)

form_login_button.click()

print("ログインボタンをクリックしました")


# ============================================================
# 8. /homeへの遷移を待つ
# ============================================================

wait.until(
    EC.url_contains("/home")
)

print("ログイン成功")
print("現在のURL:", driver.current_url)


# ============================================================
# 9. プロフィールページへ移動
# ============================================================

driver.get(
    "https://terakoya.sejuku.net/account/profile"
)

wait.until(
    EC.title_contains("プロフィール")
)

print("プロフィールページ表示成功")

time.sleep(3)


# ============================================================
# 10. 「編集」をクリック
# ============================================================

edit_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//*[normalize-space(text())='編集']"
        )
    )
)

driver.execute_script(
    "arguments[0].scrollIntoView({block:'center'});",
    edit_button
)

time.sleep(1)

edit_button.click()

print("「編集」をクリックしました")


# ============================================================
# 11. 編集画面が表示されるまで待つ
# ============================================================

time.sleep(3)


# ============================================================
# 12. textareaを調査
# ============================================================

textareas = driver.find_elements(
    By.TAG_NAME,
    "textarea"
)

print(
    f"textareaの数: {len(textareas)}"
)


# ============================================================
# 13. 自己紹介欄を特定
# ============================================================

profile_input = None


# ------------------------------------------------------------
# 最優先：
# 「自己紹介」という文字の後にあるtextarea
# ------------------------------------------------------------

try:

    profile_input = wait.until(
        EC.presence_of_element_located(
            (
                By.XPATH,
                "//*[normalize-space(text())='自己紹介']/following::textarea[1]"
            )
        )
    )

    print("自己紹介欄を特定しました")

except:
    pass


# ------------------------------------------------------------
# 上記で取れなかった場合
# 現在の自己紹介文が入っているtextareaを探す
# ------------------------------------------------------------

if profile_input is None:

    for textarea in textareas:

        value = textarea.get_attribute(
            "value"
        )

        if value and "三豊市役所" in value:

            profile_input = textarea

            print(
                "現在の自己紹介文から自己紹介欄を特定しました"
            )

            break


# ============================================================
# 14. 見つからない場合
# ============================================================

if profile_input is None:

    print()
    print("自己紹介欄を特定できませんでした。")

    print()
    print("編集画面にある入力欄を調査します。")

    print()
    print("=" * 50)

    inputs = driver.find_elements(
        By.CSS_SELECTOR,
        "input, textarea"
    )

    for i, element in enumerate(inputs):

        print()
        print(f"--- 入力欄 {i} ---")

        print(
            "tag:",
            element.tag_name
        )

        print(
            "name:",
            element.get_attribute("name")
        )

        print(
            "placeholder:",
            element.get_attribute("placeholder")
        )

        print(
            "value:",
            element.get_attribute("value")
        )


    driver.save_screenshot(
        "edit_debug.png"
    )

    driver.quit()

    raise Exception(
        "自己紹介欄を特定できませんでした"
    )


# ============================================================
# 15. 新しい自己紹介文
# ============================================================

new_profile = (
    "プログラミング学習中です！"
    "今はスクレイピングに挑戦しています！"
)


# ============================================================
# 16. 自己紹介欄までスクロール
# ============================================================

driver.execute_script(
    "arguments[0].scrollIntoView({block:'center'});",
    profile_input
)

time.sleep(1)


# ============================================================
# 17. 現在の文章を削除
#
# clear()だけではReact系フォームで
# うまく反映されない場合があるため、
# Ctrl+A → Backspaceを使用
# ============================================================

profile_input.click()

profile_input.send_keys(
    Keys.CONTROL,
    "a"
)

profile_input.send_keys(
    Keys.BACKSPACE
)


# ============================================================
# 18. 新しい自己紹介文を入力
# ============================================================

profile_input.send_keys(
    new_profile
)

print()
print("自己紹介文を変更しました")

print(
    "新しい自己紹介:",
    new_profile
)


# ============================================================
# 19. 更新・保存ボタンを探す
# ============================================================

save_button = None


# 「更新」
try:

    save_button = driver.find_element(
        By.XPATH,
        "//button[contains(normalize-space(.),'更新')]"
    )

except:
    pass


# 「保存」
if save_button is None:

    try:

        save_button = driver.find_element(
            By.XPATH,
            "//button[contains(normalize-space(.),'保存')]"
        )

    except:
        pass


# type=submit
if save_button is None:

    try:

        save_button = driver.find_element(
            By.CSS_SELECTOR,
            "button[type='submit']"
        )

    except:
        pass


# ============================================================
# 20. ボタンが見つからない場合
# ============================================================

if save_button is None:

    print()
    print("保存・更新ボタンを特定できませんでした")

    buttons = driver.find_elements(
        By.TAG_NAME,
        "button"
    )

    print()
    print("画面上のボタン:")

    for i, button in enumerate(buttons):

        print(
            i,
            repr(button.text)
        )


    driver.save_screenshot(
        "save_debug.png"
    )

    driver.quit()

    raise Exception(
        "保存ボタンを特定できませんでした"
    )


# ============================================================
# 21. 保存ボタンまでスクロール
# ============================================================

driver.execute_script(
    "arguments[0].scrollIntoView({block:'center'});",
    save_button
)

time.sleep(1)


# ============================================================
# 22. 保存
# ============================================================

save_button.click()

print("更新ボタンをクリックしました")


# ============================================================
# 23. 更新処理を待つ
# ============================================================

time.sleep(5)


# ============================================================
# 24. 更新結果を確認
# ============================================================

body_text = driver.find_element(
    By.TAG_NAME,
    "body"
).text


print()
print("=" * 60)
print("実行結果")
print("=" * 60)


if new_profile in body_text:

    print()
    print("【成功】自己紹介文が更新されました！")

else:

    print()
    print(
        "更新後の画面で指定文章を確認できませんでした。"
    )


print()
print("設定した文章:")
print(new_profile)


# ============================================================
# 25. 証拠用スクリーンショット
# ============================================================

driver.save_screenshot(
    "profile_updated.png"
)

print()
print(
    "profile_updated.png を保存しました"
)


# ============================================================
# 26. Chrome終了
# ============================================================

driver.quit()

print()
print("処理完了")


Chrome起動成功
ログイン画面を開きました
メールアドレスを入力してください: wisely1015@gmail.com
パスワードを入力してください: ··········
ログインボタンをクリックしました
ログイン成功
現在のURL: https://terakoya.sejuku.net/home
プロフィールページ表示成功
「編集」をクリックしました
textareaの数: 2
自己紹介欄を特定しました

自己紹介文を変更しました
新しい自己紹介: プログラミング学習中です！今はスクレイピングに挑戦しています！
更新ボタンをクリックしました

実行結果

【成功】自己紹介文が更新されました！

設定した文章:
プログラミング学習中です！今はスクレイピングに挑戦しています！

profile_updated.png を保存しました

処理完了
